# PFE ML — V1 Artifact Check

This notebook **verifies the deployable V1 continuity-risk artifact** — it does not score companies (use `pfe_ml_model_interrogation.ipynb` for that).

The "final good" V1 artifact is the single `model.joblib` at the root of `ml-artifacts/`, written by `train_continuity_model.py`. This notebook answers four questions:

1. **Is it there?** Locate `model.joblib` + `model_metadata.json` and show size / last-modified.
2. **Does it load?** Unpickle the bundle (handling the `__main__` custom-class gotcha) and inspect its structure.
3. **How good is it?** Read the deployed metrics (AP, AUC, precision/recall/F1).
4. **Is it the *best* run or just the *latest*?** The training script keeps only one `model.joblib` (the most recent run) and archives no per-run joblib — so the live file is the newest run, which may not be the best. We cross-check against `model_run_comparison.csv`.
5. **Does it actually predict?** A small load-and-score smoke test against real feature rows.

Storage root used by the training notebooks: `/content/drive/MyDrive/PFE ML Data/pfe_data`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = Path('/content/pfein')
BACKEND_DIR = REPO_DIR / 'back_end'

DRIVE_ROOT = Path('/content/drive/MyDrive/PFE ML Data/pfe_data')
DUCKDB_TMP = Path('/content/pfein_duckdb_tmp')
INSTALL_REQUIREMENTS = True

cwd = Path.cwd()
if (cwd / 'collabs' / 'requirements-colab.txt').exists():
    BACKEND_DIR = cwd
    REPO_DIR = BACKEND_DIR.parent

if not (BACKEND_DIR / 'collabs' / 'requirements-colab.txt').exists():
    if not (REPO_DIR / '.git').exists():
        subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'switch', BRANCH])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])

if not (BACKEND_DIR / 'collabs' / 'requirements-colab.txt').exists():
    raise FileNotFoundError(f'Backend repository is incomplete: {BACKEND_DIR}')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DUCKDB_TMP.mkdir(parents=True, exist_ok=True)
os.environ['DUCKDB_TEMP_DIRECTORY'] = str(DUCKDB_TMP)

os.chdir(BACKEND_DIR)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

if INSTALL_REQUIREMENTS:
    requirements = BACKEND_DIR / 'collabs' / 'requirements-colab.txt'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])

print(f'BACKEND_DIR = {BACKEND_DIR}')
print(f'DRIVE_ROOT  = {DRIVE_ROOT}')
print(f'BRANCH      = {BRANCH}')

## 1. Locate the V1 artifact

The deployable file is `ml-artifacts/model.joblib`. `train_continuity_model.py` overwrites it on every run and writes its companion `model_metadata.json` alongside. The two `*_run_*` files track the history of all runs.

In [ ]:
import json
import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

ARTIFACTS_DIR = DRIVE_ROOT / 'ml-artifacts'
MODEL_PATH = ARTIFACTS_DIR / 'model.joblib'
METADATA_PATH = ARTIFACTS_DIR / 'model_metadata.json'
COMPARISON_PATH = ARTIFACTS_DIR / 'model_run_comparison.csv'
RUN_INDEX_PATH = ARTIFACTS_DIR / 'model_run_index.jsonl'

print('Artifacts directory:', ARTIFACTS_DIR, '(exists:', ARTIFACTS_DIR.exists(), ')\n')

expected = {
    'model.joblib  (the deployable artifact)':        MODEL_PATH,
    'model_metadata.json  (metrics for that file)':   METADATA_PATH,
    'model_run_comparison.csv  (all runs)':           COMPARISON_PATH,
    'model_run_index.jsonl  (append-only run log)':   RUN_INDEX_PATH,
}
rows = []
for label, p in expected.items():
    exists = p.exists()
    rows.append({
        'artifact': label,
        'exists': exists,
        'size_MB': round(p.stat().st_size / 1e6, 3) if exists else None,
        'last_modified_utc': datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc).isoformat(timespec='seconds') if exists else None,
    })
display(pd.DataFrame(rows))

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'No deployable model at {MODEL_PATH}. Run train_continuity_model.py first.')
if not METADATA_PATH.exists():
    raise FileNotFoundError(f'No metadata at {METADATA_PATH}.')

print('\nEverything under ml-artifacts/:')
for p in sorted(ARTIFACTS_DIR.glob('*')):
    kind = 'dir' if p.is_dir() else 'file'
    print(f'  [{kind}] {p.name}')

## 2. Load the artifact (the `__main__` gotcha)

The pipeline was pickled while `train_continuity_model.py` ran as `__main__`, so it references `__main__.CategoricalCardinalityCapper` (and other custom classes). Importing the module is not enough — pickle resolves those names on `__main__` specifically. We re-alias every class from the training module onto `__main__` before `joblib.load`. The saved object is a **bundle dict**, not a bare estimator; the model is under `bundle['pipeline']`.

In [ ]:
import joblib

# Re-alias the training module's custom classes onto __main__ so unpickling resolves them.
from app.tools import train_continuity_model as _tcm
_main = sys.modules['__main__']
for _name in dir(_tcm):
    _obj = getattr(_tcm, _name)
    if isinstance(_obj, type) and getattr(_obj, '__module__', None) == _tcm.__name__:
        setattr(_main, _name, _obj)

bundle = joblib.load(MODEL_PATH)
if not (isinstance(bundle, dict) and 'pipeline' in bundle):
    raise TypeError(f'Expected a bundle dict with a "pipeline" key; got {type(bundle).__name__}')

pipeline = bundle['pipeline']
feature_columns = list(bundle.get('feature_columns') or [])
numeric_columns = list(bundle.get('numeric_columns') or [])
categorical_columns = list(bundle.get('categorical_columns') or [])

print('Loaded OK.\n')
print('Bundle keys   :', sorted(bundle.keys()))
print('Model version :', bundle.get('model_version'))
print('Target        :', bundle.get('target'))
print('Horizon months:', bundle.get('horizon_months'))
print('Trained at    :', bundle.get('trained_at'))
print(f'Feature count : {len(feature_columns)}  ({len(numeric_columns)} numeric + {len(categorical_columns)} categorical)')
print()
print('Pipeline steps:')
for step_name, step in pipeline.named_steps.items():
    print(f'  - {step_name}: {type(step).__name__}')

## 3. Deployed metrics

These are the held-out test metrics for the file currently saved as `model.joblib`, read from `model_metadata.json`. For this rare target, **average precision (AP)** is the headline number — accuracy is misleading at a ~4% base rate.

In [ ]:
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
m = metadata.get('metrics', {})

print('=== Run identity ===')
print('model_version   :', metadata.get('model_version'))
print('run_name        :', metadata.get('run_name'))
print('model_family    :', metadata.get('model_family'))
print('train years     :', metadata.get('train_start_year'), '->', metadata.get('train_end_year'))
print('split strategy  :', metadata.get('split_strategy'))
print('rows train/test :', metadata.get('train_rows'), '/', metadata.get('test_rows'))
print('sample strategy :', metadata.get('sample_strategy'))
print('feature count   :', metadata.get('feature_count'))

print('\n=== Headline metrics (held-out test) ===')
headline = ['average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5', 'accuracy']
for k in headline:
    v = m.get(k)
    print(f'  {k:20s}: ' + (f'{v:.4f}' if isinstance(v, (int, float)) else str(v)))

# Base rate / class balance on the test split.
test_counts = metadata.get('test_class_counts') or {}
pos = test_counts.get('1') or test_counts.get(1)
tot = sum(int(v) for v in test_counts.values()) if test_counts else None
if pos is not None and tot:
    base = int(pos) / tot
    ap = m.get('average_precision')
    print(f'\n  positive rate (test): {base:.4%}  ({pos} / {tot})')
    if isinstance(ap, (int, float)) and base:
        print(f'  AP lift over base rate: {ap / base:.1f}x')

ta = m.get('threshold_analysis')
if ta:
    print('\nThreshold analysis:')
    display(pd.DataFrame(ta))
tk = m.get('top_k_analysis')
if tk:
    print('Top-K analysis (operating points for an action-taker):')
    display(pd.DataFrame(tk))

## 4. Is the deployed file the *best* run, or just the *latest*?

`train_continuity_model.py` writes only one `model.joblib` (overwritten each run) and does **not** archive a joblib per run — so the live file is always the **most recent** training run, which is not necessarily the best-scoring one. We rank every recorded run in `model_run_comparison.csv` (restricted to the deployed model's temporal split for a fair comparison) and check whether the deployed run sits at the top.

If a better run exists, the only way to deploy it is to **re-run training with that configuration** — there is no archived joblib to copy back.

In [ ]:
deployed_version = str(metadata.get('model_version'))
deployed_run = str(metadata.get('run_name'))

if not COMPARISON_PATH.exists():
    print('No model_run_comparison.csv — only one run on record, so the deployed file is trivially the best.')
else:
    comp = pd.read_csv(COMPARISON_PATH)
    metric = 'average_precision' if 'average_precision' in comp.columns else 'roc_auc'

    # Fair comparison: same temporal split as the deployed model.
    pool = comp.copy()
    dep_split = metadata.get('split_strategy')
    if 'split_strategy' in comp.columns and dep_split:
        same_split = comp[comp['split_strategy'] == dep_split]
        if not same_split.empty:
            pool = same_split
            print(f'Comparing within split_strategy == {dep_split!r} ({len(pool)} of {len(comp)} runs).')

    pool = pool.sort_values(metric, ascending=False).reset_index(drop=True)
    show_cols = [c for c in ['model_version', 'run_name', 'model_family', 'split_strategy',
                             'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'trained_at']
                 if c in pool.columns]
    print(f'\nAll runs ranked by {metric} (best first):')
    display(pool[show_cols])

    best = pool.iloc[0]
    is_best = (str(best.get('model_version')) == deployed_version) or (str(best.get('run_name')) == deployed_run)
    print(f'Best run     : {best.get("run_name")}  | {metric} = {float(best.get(metric)):.4f}')
    print(f'Deployed run : {deployed_run}  | {metric} = {float(m.get(metric)):.4f}')

    if is_best:
        print('\n[OK] The deployed model.joblib IS the best run on its split. Use it directly.')
    else:
        gap = float(best.get(metric)) - float(m.get(metric) or 0)
        print(f'\n[!] The deployed model.joblib is NOT the best run ({metric} gap = {gap:+.4f}).')
        print('    No per-run joblib is archived, so to deploy the better run above you must re-run')
        print(f'    train_continuity_model.py with that run\'s configuration (family/params/split).')

## 5. Load + predict smoke test

Pull a small sample of real feature rows, align them to the exact columns the pipeline expects, and run `predict_proba`. This confirms the artifact is *usable end-to-end*, not just loadable. We use `LIMIT` (not a full reservoir sample) so it stays fast — the goal is "does scoring run without error and produce sane probabilities", not a representative distribution (Section 3 already has the held-out metrics for that).

In [ ]:
import duckdb
import numpy as np

DATA_LAKE = DRIVE_ROOT / 'data-lake'
FEATURES_PATH = DATA_LAKE / 'features' / 'company_year_features'

if not FEATURES_PATH.exists() or not any(FEATURES_PATH.rglob('*.parquet')):
    print('No feature parquet at', FEATURES_PATH)
    print('Skipping the predict smoke test — the artifact still loaded and inspected fine above.')
else:
    SAMPLE_ROWS = 5000
    glob_sql = (FEATURES_PATH / '**' / '*.parquet').as_posix().replace("'", "''")
    con = duckdb.connect()
    try:
        sample = con.execute(
            f"SELECT * FROM read_parquet('{glob_sql}', union_by_name=true) LIMIT {SAMPLE_ROWS}"
        ).df()
    finally:
        con.close()
    print(f'Sampled {len(sample)} feature rows.')

    # Align to the exact training columns: add any missing as NaN, cast bools to float, reorder.
    X = sample.copy()
    missing = [c for c in feature_columns if c not in X.columns]
    for c in missing:
        X[c] = np.nan
    for c in X.columns:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype(float)
    X = X[feature_columns]
    if missing:
        print('Columns absent from the parquet (added as NaN):', missing)

    scores = pipeline.predict_proba(X)[:, 1]
    s = pd.Series(scores, name=bundle.get('target', 'continuity_risk_12m_score'))
    print('\n[OK] predict_proba ran on', len(s), 'rows.')
    print(f'  min={s.min():.4f}  median={s.median():.4f}  mean={s.mean():.4f}  max={s.max():.4f}  NaN={int(s.isna().sum())}')
    display(s.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame())

## 6. Confirm the censoring hypothesis (by-year diagnostic)

Section 3 showed AP 0.153 on the 2024 test year — well below the ~0.30 baseline tested on 2023. The hypothesis is **label right-censoring**: the `continuity_risk_12m` label for a Dec-2024 cutoff needs events through late 2025, which aren't fully ingested yet, so 2024 positives are under-counted.

This cell tests that directly: positive rate by `prediction_year` over the full labels, plus a sampled AP/AUC per year. If the latest year(s) show a sharp drop in **both** positive rate and AP, censoring is confirmed and you should headline the most recent **mature** year — not 2024.

In [ ]:
import duckdb
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

TARGET = bundle.get('target', 'continuity_risk_12m_label')
LABELS_PATH = DATA_LAKE / 'features' / 'risk_labels'

have_features = FEATURES_PATH.exists() and any(FEATURES_PATH.rglob('*.parquet'))
have_labels = LABELS_PATH.exists() and any(LABELS_PATH.rglob('*.parquet'))

if not have_labels:
    print('No risk_labels parquet at', LABELS_PATH, '- cannot run the by-year diagnostic.')
else:
    label_glob = (LABELS_PATH / '**' / '*.parquet').as_posix().replace("'", "''")

    # 1) Positive rate by year over the FULL labels (unbiased, cheap). This alone
    #    shows censoring: a sharp drop on the latest year(s) means the 12-month
    #    forward window is only partially observed.
    con = duckdb.connect()
    try:
        rate = con.execute(f'''
            SELECT prediction_year,
                   COUNT(*)                                  AS n,
                   SUM(CAST("{TARGET}" AS INTEGER))          AS positives,
                   AVG(CAST("{TARGET}" AS DOUBLE))           AS positive_rate
            FROM read_parquet('{label_glob}', union_by_name=true)
            WHERE "{TARGET}" IS NOT NULL
            GROUP BY prediction_year
            ORDER BY prediction_year
        ''').df()
    finally:
        con.close()
    print(f'Positive rate of {TARGET} by prediction_year (full labels):')
    display(rate)

    # 2) Sampled AP / AUC by year: score a per-year sample with the loaded model.
    #    AP tracks the positive rate, so AP and positive_rate should fall together
    #    on a censored year. (Sample via LIMIT for speed -- the table above is the
    #    unbiased signal; this just shows the model's AP trend.)
    if have_features:
        feat_glob = (FEATURES_PATH / '**' / '*.parquet').as_posix().replace("'", "''")
        PER_YEAR_CAP = 60000
        out = []
        for yr in rate['prediction_year'].dropna().astype(int).tolist():
            con = duckdb.connect()
            try:
                df = con.execute(f'''
                    SELECT f.*, l."{TARGET}" AS _y
                    FROM read_parquet('{feat_glob}', union_by_name=true) f
                    JOIN read_parquet('{label_glob}', union_by_name=true) l
                      USING (siren, prediction_year)
                    WHERE f.prediction_year = {yr} AND l."{TARGET}" IS NOT NULL
                    LIMIT {PER_YEAR_CAP}
                ''').df()
            finally:
                con.close()
            if df.empty:
                continue
            y = df['_y'].astype(int).to_numpy()
            X = df.copy()
            for c in [c for c in feature_columns if c not in X.columns]:
                X[c] = np.nan
            for c in X.columns:
                if pd.api.types.is_bool_dtype(X[c]):
                    X[c] = X[c].astype(float)
            proba = pipeline.predict_proba(X[feature_columns])[:, 1]
            pos = int(y.sum())
            out.append({
                'prediction_year': yr,
                'sampled_rows': len(df),
                'positives': pos,
                'positive_rate': float(y.mean()),
                'AP': float(average_precision_score(y, proba)) if pos > 0 else None,
                'AUC': float(roc_auc_score(y, proba)) if 0 < pos < len(y) else None,
            })
        ap_by_year = pd.DataFrame(out)
        print('\nSampled AP / AUC by year (LIMIT sample per year):')
        display(ap_by_year)
        print('\nRead: if positive_rate AND AP both collapse on the latest year(s),')
        print('that is label right-censoring (incomplete forward window) -- not the model degrading.')
        print('Report the most recent year with a MATURE positive rate as your V1 headline.')

## Verdict & how to use it

If all five sections passed, the file at `ml-artifacts/model.joblib` is your **final good V1 artifact**. To use it:

- **Score companies** → use `pfe_ml_model_interrogation.ipynb` (loads this same file, scores by SIREN, adds the post-cutoff rules layer).
- **Serve via the FastAPI backend** → copy this `model.joblib` into `back_end/app/ml/artifacts/model.joblib`. The backend loader (`app/ml/loader.py`) reads exactly `ML_ARTIFACTS_DIR / ML_MODEL_FILE`, so it picks it up on startup with no code changes.
- **Inference in your own code** → `b = joblib.load(path); b['pipeline'].predict_proba(X[b['feature_columns']])[:, 1]` (after the `__main__` re-aliasing in Section 2).

Caveats to keep in mind:
- **Raw probabilities are inflated** — the model uses class balancing for a rare target, so scores are best treated as a *ranking* signal unless you load an isotonic/Platt calibrator (Phase E).
- **The score is blind to anything after its `prediction_year` cutoff.** For live decisions, layer on the post-cutoff rules (see the interrogation notebook).
- **If Section 4 flagged a better run**, the live file is only the *latest* run — re-train that config to deploy the best one.